# LightBanglaHC: new LR comparison and artifact checks

This is a **new, unexecuted companion notebook**, not an edit to the supplied historical notebook. The actual revision experiment is recorded in `audit/lr_comparison/results.json` and paired NPZs when complete. Download and unzip the full repository, then run this notebook from it. The full six-candidate nested comparison remains a prospective protocol in S2. Do not claim that this notebook has completed it.


In [ ]:
from pathlib import Path
import sys, json, platform
import numpy, pandas, scipy, sklearn
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/herwill_2026/train.csv').exists()), None)
assert ROOT is not None, 'Run from the extracted repository; use only the three-class 2026 CSVs.'
REV = ROOT/'revision_2026-10-11'
print({m.__name__:m.__version__ for m in [numpy,pandas,scipy,sklearn]})
print('Python', platform.python_version())


## Direct checks

Regenerate three stratified shuffled seed-42 folds; count raw exact-text groups, conflicts and cross-fold groups. No training or hidden test labels are involved. This writes a fresh local check report.


In [ ]:
import runpy
runpy.run_path(str(REV/'scripts/check_revision_evidence.py'), run_name='__main__')


## Full-data paired LR experiment

Run all four source TF-IDF branches and LR_C3 settings. The two arms hold global class weights fixed to isolate TF-IDF scope. The wall budget below is four hours; reduce `--jobs` if memory is limited. This reruns from scratch and may take substantial time. Convergence warnings must be retained. It trains on CPU and does not need PyTorch/THOP.


In [ ]:
saved_argv = sys.argv[:]
try:
    sys.argv = ['lr_preprocessing_comparison.py', '--wall-seconds', '14400', '--jobs', '3', '--out', str(REV/'audit/lr_rerun')]
    runpy.run_path(str(REV/'scripts/lr_preprocessing_comparison.py'), run_name='__main__')
finally:
    sys.argv = saved_argv


In [ ]:
result=json.loads((REV/'audit/lr_rerun/results.json').read_text())
if result['status']=='completed' and all(a['completed'] for a in result['arms'].values()):
    print('Global F1',result['arms']['global']['pooled_macro_f1'])
    print('Fold-fitted F1',result['arms']['within_fold']['pooled_macro_f1'])
    print('Global minus fold-fitted',result['global_minus_within_fold'])
    print('Inspect all n_iter and convergence flags:',result['arms'])
else:
    print('Incomplete; no paired result may be claimed.',result)


## Full stacking protocol

This cell displays the protocol; it does **not** execute the six-candidate nested experiment. A matched original-versus-corrected comparison must protect outer labels and explicitly state its prediction/averaging policy.


In [ ]:
from IPython.display import display, Markdown
display(Markdown((REV/'docs/STACKING_PROTOCOL.md').read_text()))
